# 第14章　计算图与自动微分 —— 反向传播的工程本质

> **本章目标**：理解计算图（节点=操作，边=数据流），用 PyTorch 的 `.backward()` 和 `torch.no_grad()` 掌控梯度流动。
> **前置知识**：第 13 章（链式法则）、第 6 章（矩阵乘法）

In [ ]:
import torch
import numpy as np
print("✅ 环境就绪")

---
## 14.1~14.3　计算图与反向传播

📐 **计算图（Computational Graph）**：节点=操作，边=数据。前向从左到右，反向从右到左传播梯度。

y = (x+2)×3：`x → [+2] → u → [×3] → y`。∂y/∂x = 3。

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
u = x + 2
y = u * 3
y.backward()
print(f"y = {y.item()},   dy/dx = {x.grad.item()}")  # 9, 3

# sin(x^2) at x=2
x2 = torch.tensor(2.0, requires_grad=True)
y2 = torch.sin(x2**2)
y2.backward()
print(f"sin(x^2) at x=2: grad = {x2.grad.item():.4f}  (cos(4)*4 = {torch.cos(torch.tensor(4.))*4:.4f})")

---
## 14.6　切断梯度：detach() 与 no_grad()

📐 **`.detach()`**：创建副本，断开计算图。
📐 **`torch.no_grad()`**：整个代码块不建图（用于推理）。

In [ ]:
import time
x = torch.randn(1000, 1000)

t0 = time.perf_counter()
for _ in range(200):
    y = x @ x.T; y.sum()
t1 = time.perf_counter()

with torch.no_grad():
    t2 = time.perf_counter()
    for _ in range(200):
        y = x @ x.T; y.sum()
    t3 = time.perf_counter()

print(f"with grad: {t1-t0:.3f}s   no_grad: {t3-t2:.3f}s   speedup: {(t1-t0)/(t3-t2):.1f}x")

---
## ✏️ 习题

> 在下方新建代码单元格作答。

1. （概念）计算图中节点和边分别代表什么？
2. （概念）`.detach()` 和 `torch.no_grad()` 有什么区别？
3. （代码）对 y=(x×2+1)^2 在 x=2 处，手算反向传播各节点梯度，用 PyTorch 验证。
4. （代码）对比有/无 `no_grad()` 下循环 1000 次矩阵乘法的耗时。

---

> 🔗 **章末钩子**：你已经能操控梯度了。但梯度是从"不确定性"中产生的——如果模型 100% 确定答案，梯度就是零。
> 预览：下一章——**概率分布**，从掷骰子到正态分布。

> 💡 **提示**：完成后运行 `Kernel → Restart & Run All` 验证。